MOBILE MONEY LIQUIDITY STRESS PREDICTION

 1. Introduction

This project develops an end-to-end machine learning solution to predict whether a mobile money customer will experience liquidity stress within the next 30 days.

 2. Problem Statement

The objective is to develop a binary classification model using customer demographics, transaction behaviour, activity rates and account balance information.

3. Machine Learning Objective

Target Variable: liquidity_stress_next_30d

Class 0: No liquidity stress
Class 1: Liquidity stress

In [32]:
#import my libraries
import numpy as np
import pandas as pd

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

In [28]:
#load dataset
df = pd.read_csv("Train (1).csv")

In [29]:
#Explore dataset, display first five of dataset

df.head()

,arpu,age,gender,region,smartphone,segment,earning_pattern,x_90_d_activity_rate,m1_paybill_volume,m1_paybill_total_value,...,m6_withdraw_highest_amount,m6_withdraw_agents,m1_daily_avg_bal,m2_daily_avg_bal,m3_daily_avg_bal,m4_daily_avg_bal,m5_daily_avg_bal,m6_daily_avg_bal,liquidity_stress_next_30d,ID
0,2653.88,47,F,Central Zone,Yes,MVC,Monthly Earner,0.6905,8,58372.55,...,0.00,0.0,12488.72,11404.86,12943.70,14157.61,13202.11,4336.98,1,ID_C1DF84C4D2
1,2653.88,47,F,Central Zone,Yes,MVC,Monthly Earner,0.6905,0,0.00,...,5191.14,4.0,13677.61,2781.25,5897.59,5416.23,11062.23,4849.68,0,ID_CB8B764D76
2,2653.88,47,F,Central Zone,Yes,MVC,Monthly Earner,0.6905,12,41262.86,...,0.00,0.0,7271.09,5364.41,9023.13,13001.02,17107.92,15174.32,1,ID_D40D58FC67
3,2653.88,47,F,Central Zone,Yes,MVC,Monthly Earner,0.6905,10,8738.58,...,8965.63,3.0,14331.10,33465.26,2973.45,3577.07,18990.91,8488.26,0,ID_CFD8F26581
4,20535.82,63,F,Northern Highlands,No,MVC,Weekly Earner,0.8959,2,3654.68,...,0.00,0.0,5602.75,8497.49,26252.75,7618.93,9290.65,4214.71,0,ID_20EB501231


In [30]:
df.shape

(40000, 184)

In [12]:
df.info

<bound method DataFrame.info of            arpu  age gender              region smartphone segment  \
0       2653.88   47      F        Central Zone        Yes     MVC   
1       2653.88   47      F        Central Zone        Yes     MVC   
2       2653.88   47      F        Central Zone        Yes     MVC   
3       2653.88   47      F        Central Zone        Yes     MVC   
4      20535.82   63      F  Northern Highlands         No     MVC   
...         ...  ...    ...                 ...        ...     ...   
39995   1554.59   30      F  Southern Highlands         No     LVC   
39996  26548.53   57      M            Islander         No     MVC   
39997  26548.53   57      M            Islander         No     MVC   
39998  26548.53   57      M            Islander         No     MVC   
39999  26548.53   57      M            Islander         No     MVC   

      earning_pattern  x_90_d_activity_rate  m1_paybill_volume  \
0      Monthly Earner                0.6905                  

In [14]:
df.describe()

,arpu,age,x_90_d_activity_rate,m1_paybill_volume,m1_paybill_total_value,m1_paybill_highest_amount,m1_paybill_companies,m2_paybill_volume,m2_paybill_total_value,m2_paybill_highest_amount,...,m6_withdraw_total_value,m6_withdraw_highest_amount,m6_withdraw_agents,m1_daily_avg_bal,m2_daily_avg_bal,m3_daily_avg_bal,m4_daily_avg_bal,m5_daily_avg_bal,m6_daily_avg_bal,liquidity_stress_next_30d
count,40000.000000,40000.00000,40000.000000,40000.000000,4.000000e+04,40000.000000,40000.000000,40000.000000,4.000000e+04,40000.000000,...,4.000000e+04,40000.000000,40000.000000,40000.000000,40000.000000,40000.000000,40000.000000,40000.000000,40000.000000,40000.000000
mean,24480.982653,44.20050,0.504210,3.178500,3.355774e+04,3463.667285,2.127475,3.215975,3.366216e+04,3520.387921,...,1.028221e+05,9297.774969,1.977875,14684.315477,14610.007011,14634.600378,14711.700646,14809.994068,14684.074900,0.150000
std,51992.965165,15.32212,0.229379,4.964303,1.246310e+05,8290.539405,2.743577,5.029469,1.268042e+05,8465.908039,...,3.555190e+05,19146.708926,1.832040,32739.081130,32250.370737,32287.777394,32675.269386,33056.285393,32921.604867,0.357076
min,100.920000,18.00000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000e+00,0.000000,...,0.000000e+00,0.000000,0.000000,139.520000,137.820000,158.150000,97.330000,139.520000,163.040000,0.000000
25%,2809.712500,31.00000,0.332675,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000e+00,0.000000,...,0.000000e+00,0.000000,0.000000,1445.915000,1441.682500,1438.710000,1454.952500,1438.920000,1439.532500,0.000000
50%,7070.795000,44.00000,0.508450,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000e+00,0.000000,...,6.141555e+03,2025.410000,2.000000,5082.590000,5076.255000,5080.305000,5066.015000,5072.920000,5060.740000,0.000000
75%,21021.690000,57.00000,0.667325,5.000000,1.270061e+04,3431.962500,4.000000,5.000000,1.282144e+04,3436.347500,...,4.930610e+04,10361.752500,4.000000,11544.090000,11478.565000,11502.472500,11508.355000,11518.010000,11456.060000,0.000000
max,299815.190000,70.00000,1.000000,41.000000,2.987089e+06,49998.420000,8.000000,38.000000,4.202311e+06,49997.310000,...,8.153060e+06,112480.440000,5.000000,529537.290000,667242.510000,516521.790000,539574.450000,499853.500000,486015.760000,1.000000


In [18]:
df.columns

Index(['arpu', 'age', 'gender', 'region', 'smartphone', 'segment',
       'earning_pattern', 'x_90_d_activity_rate', 'm1_paybill_volume',
       'm1_paybill_total_value',
       ...
       'm6_withdraw_highest_amount', 'm6_withdraw_agents', 'm1_daily_avg_bal',
       'm2_daily_avg_bal', 'm3_daily_avg_bal', 'm4_daily_avg_bal',
       'm5_daily_avg_bal', 'm6_daily_avg_bal', 'liquidity_stress_next_30d',
       'ID'],
      dtype='object', length=184)

In [23]:
df.value_counts()

arpu       age  gender  region                      smartphone  segment  earning_pattern   x_90_d_activity_rate  m1_paybill_volume  m1_paybill_total_value  m1_paybill_highest_amount  m1_paybill_companies  m2_paybill_volume  m2_paybill_total_value  m2_paybill_highest_amount  m2_paybill_companies  m3_paybill_volume  m3_paybill_total_value  m3_paybill_highest_amount  m3_paybill_companies  m4_paybill_volume  m4_paybill_total_value  m4_paybill_highest_amount  m4_paybill_companies  m5_paybill_volume  m5_paybill_total_value  m5_paybill_highest_amount  m5_paybill_companies  m6_paybill_volume  m6_paybill_total_value  m6_paybill_highest_amount  m6_paybill_companies  m1_merchantpay_volume  m1_merchantpay_total_value  m1_merchantpay_highest_amount  m1_merchantpay_merchants  m2_merchantpay_volume  m2_merchantpay_total_value  m2_merchantpay_highest_amount  m2_merchantpay_merchants  m3_merchantpay_volume  m3_merchantpay_total_value  m3_merchantpay_highest_amount  m3_merchantpay_merchants  m4_merchant

In [24]:
#check for number of unique
df.nunique()

arpu                          9979
age                             53
gender                           2
region                           7
smartphone                       2
                             ...  
m4_daily_avg_bal             39326
m5_daily_avg_bal             39263
m6_daily_avg_bal             39317
liquidity_stress_next_30d        2
ID                           40000
Length: 184, dtype: int64

In [26]:
#check for missing data
df.isnull().sum()

arpu                         0
age                          0
gender                       0
region                       0
smartphone                   0
                            ..
m4_daily_avg_bal             0
m5_daily_avg_bal             0
m6_daily_avg_bal             0
liquidity_stress_next_30d    0
ID                           0
Length: 184, dtype: int64

In [34]:
(df['liquidity_stress_next_30d'].value_counts())

liquidity_stress_next_30d
0    34000
1     6000
Name: count, dtype: int64

In [36]:
# the percentage of people with liquidity stress (1) vs. without (0)

(df['liquidity_stress_next_30d'].value_counts(normalize=True) * 100)


liquidity_stress_next_30d
0    85.0
1    15.0
Name: proportion, dtype: float64

In [40]:
df.dtypes

arpu                         float64
age                            int64
gender                        object
region                        object
smartphone                    object
                              ...   
m4_daily_avg_bal             float64
m5_daily_avg_bal             float64
m6_daily_avg_bal             float64
liquidity_stress_next_30d      int64
ID                            object
Length: 184, dtype: object